# Busqueda de S_max (capacidad de conservacion de Falcon)

Notebook autocontenido para obtener `S_max` = capacidad **total de conservacion normal** del
International Falcon Reservoir (estacion `08461200`). Es la unica constante del modelo del
Falcon Challenge que NO sale de las series exportadas (esas solo tienen almacenamiento *observado*,
nunca la *capacidad*).

Se usan **dos metodos independientes que se validan entre si**:

- **Metodo A (scrape):** lee el valor documentado del reporte semanal del IBWC (`storage.htm`).
- **Metodo B (derivado de datos):** lo reconstruye desde las series de la Seccion 8 del PDF,
  usando `Percentage.Conservation-Web-Telemetry@08461200`:
  `S_max = Total Storage / (Conservation% / 100)`.

Unidades: el CSV de almacenamiento esta en **TCM** (miles de m3). 1 MCM (millon de m3) = 1000 TCM.

> No modifica `Descargas.ipynb`. Escribe el resultado en `data/falcon_reservoir_constants.json`.


In [ ]:
from pathlib import Path
import json
import re
import glob

import pandas as pd
import requests

DATA_DIR = Path('data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

MCM_TO_TCM = 1000.0  # 1 millon de m3 = 1000 mil m3

REPORT_URLS = {
    'weekly': 'https://ibwcsftpstg.blob.core.windows.net/wad/WeeklyReports/storage.htm',
    'daily':  'https://ibwcsftpstg.blob.core.windows.net/wad/DailyReports/flowdata.htm',
}

http = requests.Session()
http.headers.update({
    'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                   '(KHTML, like Gecko) Chrome/125 Safari/537.36'),
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9,es;q=0.8',
    'Referer': 'https://www.ibwc.gov/',
})


def html_to_text(html: str) -> str:
    """Quita estilos/scripts/tags y colapsa espacios (los reportes son HTML exportado de Excel)."""
    html = re.sub(r'<style.*?</style>', ' ', html, flags=re.S | re.I)
    html = re.sub(r'<script.*?</script>', ' ', html, flags=re.S | re.I)
    txt = re.sub(r'<[^>]+>', ' ', html)
    txt = txt.replace('&nbsp;', ' ').replace('&amp;', '&')
    return re.sub(r'\s+', ' ', txt).strip()

## Metodo A - scrape del reporte semanal (valor documentado)

El reporte lista **Amistad primero y Falcon despues**, y el titulo dice "Amistad and Falcon Dams".
Por eso hay que anclar en el encabezado de seccion `Falcon Dam Water Surface Elevation` (no en
`Falcon Dam` a secas) para no tomar el `Total Conservation` de Amistad (3,980.096).
Esperado: **3,288.726 MCM**.


In [ ]:
def scrape_smax_weekly() -> float:
    txt = html_to_text(http.get(REPORT_URLS['weekly'], timeout=60).text)
    # Anclar en el encabezado de la SECCION Falcon ("Falcon Dam Water Surface Elevation"),
    # NO en "Falcon Dam" a secas: el titulo del reporte dice "Amistad and Falcon Dams" antes
    # de la tabla de Amistad, y eso haria capturar el Total Conservation de Amistad (3,980.096).
    m = re.search(r'Falcon Dam Water Surface Elevation.*?Total Conservation\*?\s*([\d,]+\.\d+)',
                  txt, flags=re.S)
    if not m:
        raise RuntimeError('No se encontro "Total Conservation" de Falcon en el reporte semanal')
    return float(m.group(1).replace(',', ''))


s_max_mcm_A = scrape_smax_weekly()
s_max_tcm_A = s_max_mcm_A * MCM_TO_TCM
print(f'Metodo A (reporte semanal): S_max = {s_max_mcm_A:,.3f} MCM = {s_max_tcm_A:,.0f} TCM')

### Cross-check con el reporte diario

Fila `FALCON - 08461200`: columnas `Normal` (capacidad de conservacion) y `Flood Control Capacity`.
Sirve para confirmar el valor semanal y de paso capturar la capacidad de inundacion (solo contexto,
**no** la usa el SRS).


In [ ]:
def scrape_daily_falcon():
    txt = html_to_text(http.get(REPORT_URLS['daily'], timeout=60).text)
    # ... FALCON - 08461200 <fecha> <hora> <Normal> <Flood> <Elev> <Current> <Change> ...
    m = re.search(r'FALCON\s*-\s*08461200\s+\S+\s+\S+\s+([\d.]+)\s+([\d.]+)', txt)
    if not m:
        return None
    return {'normal_mcm': float(m.group(1)), 'flood_mcm': float(m.group(2))}


daily = scrape_daily_falcon()
flood_mcm = daily['flood_mcm'] if daily else None
print('Reporte diario Falcon:', daily)
if daily:
    assert round(s_max_mcm_A) == round(daily['normal_mcm']), 'weekly vs daily no coinciden'
    print(f'Cross-check OK: semanal {round(s_max_mcm_A)} MCM == diario {round(daily["normal_mcm"])} MCM')
    print(f'Capacidad de inundacion (contexto): {flood_mcm} MCM')

## Metodo B - derivado de las series de la Seccion 8

`S_max = Total Storage / (Conservation% / 100)`, tomando la **mediana** sobre toda la serie
(robusta a redondeo/ruido). Requiere exportar `Percentage.Conservation-Web-Telemetry@08461200`
a `data/` (igual que los otros 4 CSV). Si falta, esta celda se salta sin romper.

> La unidad del CSV de almacenamiento se **autodetecta** del header `Value (...)` (puede venir en
> `m^3`, `TCM` o `MCM` segun como se exporte del dashboard) y se normaliza a MCM antes de dividir.


In [ ]:
def find_csv(substr: str):
    hits = [p for p in glob.glob(str(DATA_DIR / '*.csv')) if substr in p]
    return hits[0] if hits else None


def load_series(path: str):
    """Devuelve (serie, unidad). La unidad sale del header 'Value (...)' del export,
    porque el dashboard permite exportar en m^3, TCM o MCM aunque el dataset se llame '...tcm'."""
    header = pd.read_csv(path, skiprows=1, nrows=0)
    um = re.search(r'\(([^)]+)\)', header.columns[1] if len(header.columns) > 1 else '')
    unit = um.group(1).strip() if um else ''
    df = pd.read_csv(path, skiprows=1)
    df.columns = ['ts', 'val']
    df = df[df['ts'].astype(str).str.match(r'\d{4}-\d{2}-\d{2}')].copy()
    df['ts'] = pd.to_datetime(df['ts'])
    df['val'] = pd.to_numeric(df['val'], errors='coerce')
    return df.dropna().set_index('ts')['val'], unit


def storage_to_mcm(series: pd.Series, unit: str) -> pd.Series:
    """Normaliza una serie de volumen a MCM (millones de m3)."""
    u = unit.lower().replace(' ', '')
    if u in ('m^3', 'm3'):
        return series / 1e6
    if u in ('tcm',):
        return series / 1e3
    if u in ('mcm',):
        return series
    raise ValueError(f'Unidad de almacenamiento no reconocida: {unit!r}')


storage_path = find_csv('Total Storage.Web-Daily-tcm@08461200')
pct_path = find_csv('Percentage.Conservation')

s_max_mcm_B = None
if storage_path and pct_path:
    storage_raw, s_unit = load_series(storage_path)
    storage_mcm = storage_to_mcm(storage_raw, s_unit)
    pct, _ = load_series(pct_path)                       # porcentaje (sin conversion)
    df = pd.concat({'storage_mcm': storage_mcm, 'pct': pct}, axis=1).dropna()
    df = df[df['pct'] > 1.0]                             # evita dividir por ruido ~0
    s_max_series_mcm = df['storage_mcm'] / (df['pct'] / 100.0)
    s_max_mcm_B = float(s_max_series_mcm.median())
    print(f'Metodo B (derivado): S_max = {s_max_mcm_B:,.3f} MCM = {s_max_mcm_B * MCM_TO_TCM:,.0f} TCM '
          f'(n={len(df)}, unidad storage={s_unit!r}, std={s_max_series_mcm.std():,.3f} MCM)')
else:
    print('Metodo B OMITIDO: falta exportar Percentage.Conservation-Web-Telemetry@08461200 a data/')
    if not storage_path:
        print('  (tampoco se encontro el CSV de Total Storage)')

## Validacion cruzada y persistencia

Se toma el valor documentado (Metodo A) como autoritativo, y si esta el Metodo B se exige que
concuerden a <1%. Resultado a `data/falcon_reservoir_constants.json`.


In [ ]:
s_max_mcm = s_max_mcm_A
s_max_tcm = s_max_mcm * MCM_TO_TCM
s_max_m3 = s_max_mcm * 1e6
s_min_tcm = 0.25 * s_max_tcm

if s_max_mcm_B is not None:
    rel = abs(s_max_mcm_B - s_max_mcm_A) / s_max_mcm_A
    print(f'Validacion cruzada: A={s_max_mcm_A:.3f} MCM, B={s_max_mcm_B:.3f} MCM, dif rel={rel:.4%}')
    assert rel < 0.01, 'Los metodos A y B difieren mas de 1%'
    print('OK: ambos metodos concuerdan (<1%)')

if storage_path:
    storage_raw, s_unit = load_series(storage_path)
    obs_max_mcm = storage_to_mcm(storage_raw, s_unit).max()
    print(f'Sanity: max almacenamiento observado {obs_max_mcm:,.1f} MCM = {obs_max_mcm / s_max_mcm:.1%} de S_max')

constants = {
    's_max_mcm': round(s_max_mcm, 3),
    's_max_tcm': round(s_max_tcm, 3),
    's_max_m3': round(s_max_m3, 1),
    's_min_tcm': round(s_min_tcm, 3),
    'flood_capacity_mcm': flood_mcm,
    'method_A_scraped_mcm': round(s_max_mcm_A, 3),
    'method_B_derived_mcm': round(s_max_mcm_B, 3) if s_max_mcm_B is not None else None,
    'source': ('IBWC WeeklyReports/storage.htm (Falcon Total Conservation) + '
               'Percentage.Conservation-Web-Telemetry@08461200'),
}
out = DATA_DIR / 'falcon_reservoir_constants.json'
out.write_text(json.dumps(constants, indent=2), encoding='utf-8')
print('\nEscrito', out)
print(json.dumps(constants, indent=2))